<a href="https://colab.research.google.com/github/adrianacalderon/CaribbeanTraining/blob/main/3_Hydrological_Persistence.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Hydrological Persistence**

---

# **Clone the HydroSOS GitHub Repository**

This tutorial presents an example of seasonal streamflow forecasting using the ***hydrological persistence*** method. In summary, this method is based on the premise that the flow anomaly observed in the most recent month will persist over the next *N* months of the forecast range.

The scripts below are based on the methodology presented in:



*   *Svensson, C. (2016). Seasonal river flow forecasts for the United Kingdom using persistence and historical analogues. Hydrological Sciences Journal, 61(1), 19-35.*

We will clone the WMO HydroSOS repository hosted on GitHub to collect a sample time series of daily streamflow.

*   Note: To use the `git clone` command on any Python notebook/script execution platform on your local machine (e.g., ***Jupyter***, ***Visual Studio Code***), you must make sure that `Git` is installed on your computer. Otherwise, Git can be downloaded from (https://git-scm.com/downloads)



In [ ]:
# Clone git respository
!git clone https://github.com/wmo-im/HydroSOS/

---

# **Import Python Libraries**

Let's import the libraries needed for this tutorial:

In [ ]:
import os
import pandas as pd
import numpy as np

from pathlib import Path
from datetime import datetime

---

# **Create new folder**

We will automatically create an `output` folder in the root directory where we will export the results:

In [ ]:
# Define folder path (e.g., in the current working directory)
folder_output = 'output'

# Create the folder if it doesn't exist
os.makedirs(folder_output, exist_ok=True)

---

# **1. Define Parameters**

Here we will define the parameters for generating the forecast based on hydrological persistence:

*   `File  path`: Path where the .csv file with daily streamflow is located
*   `fcst_range`: Range (in months) of the intended forecast
*   `apply_transformation`: Indicates whether we want to apply a logarithmic transformation to the monthly streamflow (True/False)
*   `min_valid_frac`: Minimum fraction of valid daily data in a month required to convert them to a monthly average

*   `clim_startYear`: Start year of the reference period for calculating anomalies
*   `clim_endYear`: End year of the reference period for calculating anomalies.




In [ ]:
file_path = './HydroSOS/example_data/forecast/input/obs_dir/obs_1.csv' # Path to your CSV file
fcst_range = 3                    # Forecast range (months ahead)
apply_transformation = True       # Set to True to apply log transformation on flows
min_valid_frac = 0.5              # Minimum fraction of valid daily data per month
clim_startYear = 1991             # Start year to compute anomalies (mean, standard deviation)
clim_endYear = 2020               # End year to compute anomalies

---

# **2. Read the data**

2.1. We will read the .csv file containing the daily streamflow data and convert it to monthly. The .csv file must have 2 columns (one with the dates, and another with the streamflow values), as well as:

1.   The title of the date column defined as '*Date*' (without quotes)
2.   The dates in **YYYY-MM-DD** format

In [ ]:
# Read and preprocess data
df = pd.read_csv(file_path)
df['Date'] = pd.to_datetime(df['Date'], format='%Y-%m-%d')

df.set_index('Date', inplace=True)

# Convert to monthly data
df_month = df.resample('MS').mean()

2.2. Plot streamflow

In [ ]:
#@title Plot
# Plot on screen (subplot for daily and monthly data)
from matplotlib import pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(12, 6))
axes = axes.flatten()

df.plot(ax=axes[0], color='k', linewidth=0.5, title='Daily')
df_month.plot(ax=axes[1], color='k', linewidth=0.5, title='Monthly')

for ax in axes:
  ax.set_ylabel('Streamflow (m³/s)')
  ax.legend(['Observed'])

plt.tight_layout()
plt.show()

---

# **3. Pre-processing of monthly data**


3.1. Remove the months with insufficient daily data according to the minimum defined in the parameters section:

In [ ]:
# Resample daily to monthly step and compute de fraction of dates in each month without missing data
valid_fraction = df.resample('MS').apply(lambda x: x.count() / x.size)

# Replace monthly values to NaN if % of valid < minimum
df_month.mask(valid_fraction < min_valid_frac, np.nan, inplace=True)

3.2. Applying logarithmic transformation $y = log(Q)$ to the monthly streamflow series (if this option has been enabled by the user in the parameter definitions). This process should make the distribution of streamflow more similar to a normal distribution for calculating standardized monthly anomalies:

In [ ]:
# Apply log transformation if selected
if apply_transformation:
    if (df_month <= 0).sum().values > 0:
        raise ValueError("Log transformation requires all discharge values to be > 0.")
    df_month_transformed = np.log(df_month)
else:
    df_month_transformed = df_month.copy()

3.3. Check the streamflow histogram to see the distribution::

In [ ]:
# Plot the discharge distribution in a histogram
df_month_transformed.hist()

---

# **4. Calculate standard anomalies**



4.1. Calculation of the mean $ (\overline{x}_m) $ and standard deviation $ (\sigma_m) $ for each calendar month. The statistics are calculated for the reference climatological period:

In [ ]:
# Converting date strings to datetime format
dt_start = datetime(clim_startYear, 1, 1)
dt_end = datetime(clim_endYear, 12, 31)

# Get data for the period of interest
monthly_transformed_clim = df_month_transformed[(df_month_transformed.index >= dt_start) &
                                                  (df_month_transformed.index <= dt_end)]

# Compute mean and standard deviation for each calendar month
month_nums = monthly_transformed_clim.index.month
clim_mean = monthly_transformed_clim.groupby(month_nums).mean()
clim_std = monthly_transformed_clim.groupby(month_nums).std()

4.2. Display the mean and standard deviation

In [ ]:
# For visualization purposes only
conc_data = pd.concat([clim_mean, clim_std], axis=1)
conc_data.columns = ['Average', 'Std Deviation']
conc_data.index.name = 'Month'
display(conc_data)

4.3. Calculation of standardized anomalies for each value in the monthly series. The calculation uses the mean and standard deviation of the month corresponding to the observed value, according to the following equation:  

\begin{align}
  a_{t,m} = \frac{q_{t,m}-\overline{x_m}}{\sigma_m}
\end{align}

Where:

*   $ a_{t,m} $ is the anomaly for time interval $t$ and month $m$;
*   $ q_{t,m} $ is the (transformed) streamflow for time interval $t$ and month $m$;
*   $ \overline{x_m} $ is the mean for month $m$;
*   $ \sigma_m $ is the standard deviation for month $m$.


In [ ]:
# Compute standardized anomalies
df_anomalies = df_month_transformed.copy()

# Compute anomalies for each month separately
for i in range(1, 13):
  idx = (df_anomalies.index.month == i)
  df_anomalies.loc[idx] = (df_month_transformed.loc[idx] - clim_mean.loc[i].values) / clim_std.loc[i].values

4.4. Plot the standardized anomalies for the entire monthly series.

In [ ]:
#@title Plot
# Plot on screen
df_anomalies.plot(figsize=(12, 4),
                  title=f'Standardized Monthly Anomalies, reference: {clim_startYear}-{clim_endYear}',
                  ylabel='Anomaly (-)',
                  legend=False)

---

# **5. Generating the forecast**

5.1. To generate the forecast, we will first identify the most recently observed anomaly:

In [ ]:
# Check standardized anomaly for last month
anomaly_date = df_month.index[-1]
anomaly_value = df_anomalies.loc[anomaly_date].values

print(f'anomaly month: {anomaly_date.strftime("%Y-%m")}')
print(f'verified anomaly: {anomaly_value}')

5.2. We identify the dates of the *N* forecast months and persist the observed anomaly onto the mean and standard deviation of each of them, using the equation:

\begin{align}
  q_m = \overline{x_m} + (a_o \times \sigma_m)
\end{align}

Where:

*   $ q_{m} $ is the (transformed) forecasted streamflow for month $m$;
*   $ \overline{x_m} $ is the mean for month $m$;
*   $ a_{o} $ is the anomaly observed in the most recent month;
*   $ \sigma_m $ is the standard deviation for month $m$.



In [ ]:
# Generate forecast dates
forecast_dates = pd.date_range(start=anomaly_date + pd.offsets.MonthEnd(1),
                               periods=fcst_range,
                               freq='MS')

# Get forecast months
forecast_month_nums = forecast_dates.month

# Get the climatological mean and standard deviation for these months
clim_mean_fcst = clim_mean.loc[forecast_month_nums].values
clim_std_fcst = clim_std.loc[forecast_month_nums].values

# Reconstruct forecast values by persisting anomaly on climatological mean and standard deviation
df_forecast_transformed = clim_mean_fcst + (anomaly_value * clim_std_fcst)

Back-transform the streamflow $ ( Q = e^y ) $ (in case they were previously transformed):

In [ ]:
# Back-transform if log transformation was applied
if apply_transformation:
    forecast_values = np.exp(df_forecast_transformed)
else:
    forecast_values = df_forecast_transformed

# Ensure only positive values
forecast_values[forecast_values < 0.0] = 0.0

5.4. Create the final table including dates and values obtained for the forecast:

In [ ]:
# Output forecast
df_forecast = pd.DataFrame(forecast_values,
                           index = forecast_dates,
                           columns = df_month.columns)

df_forecast.index.name = 'Date'

5.5. Plot the generated forecast:

In [ ]:
#@title Plot
# Plot on screen
fig, ax = plt.subplots(figsize=(15, 4))

df_month[-36:].plot(ax=ax, color='k') #Plotting the last 36 months
df_forecast.plot(ax=ax, color='r')
ax.set_ylabel('Streamflow (m³/s)')
ax.set_title('Seasonal forecast based on persistence')

plt.legend(['Observed', 'Forecast'])
plt.show()

# **6. Exporting the generated forecasts**

Finally, we export the results to a .csv file:

In [ ]:
# Save to CSV
file_name = Path(file_path).stem
df_forecast.to_csv(f'{folder_output}/persist_forecast_{file_name}.csv', float_format='%.2f')